# 01 — Mapping and contacts (NUDT15 / 5lpg)

Load the shipped 5lpg chain A mapping JSON, infer the SEQRES→PDB offset, and compute ligand / protein contacts from the raw 5lpg PDB. Outputs go to `examples/NUDT15_HUMAN/outputs/`.


In [ ]:
from pathlib import Path

from dualde.contacts import (
    NUDT15_LIGAND_NAMES,
    functional_residues,
    residue_contacts,
)
from dualde.mapping import (
    load_structural_mapping_json,
    read_fasta,
    select_mapping_entry,
    seqres_to_pdb_offset,
)
from dualde.paths import example_dir

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

header, uniprot_seq = read_fasta(EXAMPLE / "NUDT15_HUMAN.fasta")
print(header, len(uniprot_seq), "aa")

entries = load_structural_mapping_json(EXAMPLE / "mapping" / "5lpg_A.json")
entry = select_mapping_entry(entries, pdb_id="5lpg", chain="A", uniprot_id="NUD15_HUMAN")
raw_pdb = EXAMPLE / "pdb" / "5lpg.pdb"
offset = seqres_to_pdb_offset(str(raw_pdb), "5lpg", "A", search_dirs=[str(EXAMPLE / "pdb")])
print("SEQRES→PDB offset:", offset)
print("UniProt span:", entry["uniprot_start"], "–", entry["uniprot_end"])


In [ ]:
contacts = residue_contacts(
    str(raw_pdb),
    entry,
    chain="A",
    ligand_names=NUDT15_LIGAND_NAMES,
    distance_cutoff=8.0,
    seqres_to_pdb_offset=offset,
    include_protein=True,
)
func = functional_residues(contacts, distance_cutoff=3.5)

contacts.to_csv(OUT / "5lpg_contacts.csv", index=False)
func.to_csv(OUT / "5lpg_functional_residues.csv", index=False)

print(contacts.groupby("contact_type").size().to_string())
print("functional residues:", len(func))
print(func.head())


To rebuild the mapping from SIFTS instead of the shipped JSON, download [pdb_chain_uniprot.csv.gz](https://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/csv/pdb_chain_uniprot.csv.gz) and call `dualde.mapping.download_sifts_table` + `mapping_entry_from_sifts` (5lpg, chain A, UniProt Q9NV35).
